# TPU 环境

1. ✅ JAX 检测到 TPU 设备 :

这是最确凿的证据。JAX 库已经直接与 TPU 硬件通信，并识别出了 TpuDevice(id=0...) 。

2. ✅ PyTorch XLA 成功连接 :

显示 xla:0 ，这意味着 PyTorch 通过 XLA (Accelerated Linear Algebra) 编译器成功挂载了 TPU 核心。在 PyTorch 中，TPU 设备通常就被称为 xla 设备。

3. 关于环境变量 (❌ COLAB_TPU_ADDR) :

不用担心这个报错。新版本的 Colab 运行时或者特定的 TPU VM 架构（如 TPU v4/v5e）有时不会暴露旧版的 COLAB_TPU_ADDR 环境变量，但底层的驱动和库（如 JAX/PyTorch XLA）依然能找到硬件。

In [ ]:
# import os
# import sys
# import torch_xla

# print("=== Checking TPU Status ===")

# # 1. 检查 Colab 特有的环境变量
# if 'COLAB_TPU_ADDR' in os.environ:
#     print(f"✅ 环境变量检测到 TPU: {os.environ['COLAB_TPU_ADDR']}")
# else:
#     print("❌ 环境变量中未找到 COLAB_TPU_ADDR (如果是在本地运行而非 Colab，这很正常)")

# # 2. 检查 JAX (因为你的日志里出现了 JAX 的 TPU 警告)
# try:
#     import jax
#     try:
#         # 获取可用设备
#         devices = jax.devices()
#         # 检查是否有 TPU 设备
#         tpu_devices = [d for d in devices if 'tpu' in str(d).lower()]
#         if tpu_devices:
#             print(f"✅ JAX 检测到 TPU 设备: {tpu_devices}")
#         else:
#             print(f"⚠️ JAX 已安装，但未发现 TPU。当前设备: {devices}")
#     except RuntimeError as e:
#         print(f"⚠️ JAX 初始化错误 (可能是因为没有 TPU): {e}")
# except ImportError:
#     print("ℹ️ JAX 未安装")

# # 3. 检查 PyTorch (需要 torch_xla)
# try:
#     import torch_xla
#     import torch_xla.core.xla_model as xm
#     # 尝试获取 XLA 设备
#     dev = torch_xla.device()
#     print(f"✅ PyTorch XLA 成功连接到设备: {dev}")
# except ImportError:
#     print("ℹ️ torch_xla 未安装 (PyTorch 在 TPU 上运行通常需要此库)")
# except Exception as e:
#     print(f"⚠️ PyTorch XLA 检查失败: {e}")

# # 4. 检查 TensorFlow
# try:
#     import tensorflow as tf
#     try:
#         resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
#         tf.config.experimental_connect_to_cluster(resolver)
#         tf.tpu.experimental.initialize_tpu_system(resolver)
#         print(f"✅ TensorFlow 检测到 TPU: {resolver.master()}")
#     except ValueError:
#         print("❌ TensorFlow 未检测到 TPU")
#     except Exception as e:
#         print(f"⚠️ TensorFlow 检查出错: {e}")
# except ImportError:
#     print("ℹ️ TensorFlow 未安装")

=== Checking TPU Status ===
❌ 环境变量中未找到 COLAB_TPU_ADDR (如果是在本地运行而非 Colab，这很正常)


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


⚠️ JAX 初始化错误 (可能是因为没有 TPU): Unable to initialize backend 'tpu': UNKNOWN: TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0 (set JAX_PLATFORMS='' to automatically choose an available backend)
⚠️ PyTorch XLA 检查失败: TPU initialization failed: open(/dev/vfio/0): Device or resource busy: Device or resource busy; Couldn't open iommu group /dev/vfio/0
ℹ️ TensorFlow 未安装


如果使用的是 pipeline，要利用 TPU，通常需要显式指定 device 。但是，标准的 HF pipeline 对 TPU 的支持不如对 GPU 那么无缝（通常更推荐用 GPU 跑推理，TPU 跑训练）。
如果只是想简单测试，可以尝试：
```python
import torch_xla

# 获取 TPU 设备
device = torch_xla.device()

# 传递给 pipeline
# 注意：不是所有模型都能直接这样在 TPU 上通过 pipeline 完美运行，有时需要 optimum 库
classifier = pipeline("sentiment-analysis", model=..., device=device)
```

> 经典的 **TPU 资源冲突** 错误：`RuntimeError: TPU initialization failed... Device or resource busy`。
>
> ### 原因分析
> 在同一个 Notebook 会话中运行了 **JAX** 的代码
> *   **冲突点**：JAX 和 PyTorch (XLA) 都试图独占 TPU 硬件资源。
> *   **现状**：JAX 已经初始化并锁定了 TPU 设备（即使你没在用它跑复杂的运算）。当你随后尝试运行 `torch_xla` 代码时，PyTorch 发现 TPU 已经被占用（busy），因此抛出错误。
>
> ### 解决方案
> 需要**重启 Colab 的运行时 (Runtime)**，以释放被 JAX 占用的 TPU 句柄。
>
> 1.  **重启运行时**：
>     *   在 Colab 菜单栏点击 **Runtime (运行时)** > **Restart session (重启会话)** (或者 **Restart runtime**)。
> 2.  **不要运行 JAX 代码**：
>     *   重启后，**跳过** 之前的那个包含 `import jax` 的检测单元格。
>     *   **直接运行** 你刚刚添加的 PyTorch XLA 矩阵乘法代码。
>
> ### 为什么不能同时用？
> 目前的 Colab 环境通常只允许单一框架（Single Framework）独占 TPU。不能在一个 Python 进程中混合使用 JAX 和 PyTorch 来访问 TPU。每次切换框架都需要重启内核。
>

# 🌟GPU 环境

In [1]:
!nvidia-smi

Thu Dec  4 03:03:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print("=== Checking GPU Status ===")

if torch.cuda.is_available():
    print(f"✅ GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Device Count: {torch.cuda.device_count()}")
    
    # Create a random tensor and move it to GPU
    try:
        x = torch.rand(5, 3)
        print("\nTensor created on CPU:")
        print(x)
        
        x_gpu = x.to('cuda')
        print("\nTensor moved to GPU:")
        print(x_gpu)
        
        # Perform a simple operation
        y_gpu = x_gpu * 2
        print("\nResult of operation on GPU (x * 2):")
        print(y_gpu)
        print("\n✅ Basic GPU tensor operations successful!")
    except Exception as e:
        print(f"\n❌ Error performing GPU operations: {e}")
else:
    print("❌ GPU is NOT available. Please check your runtime settings.")

=== Checking GPU Status ===
✅ GPU is available: Tesla T4
CUDA Version: 12.6
Device Count: 1

Tensor created on CPU:
tensor([[0.7607, 0.6726, 0.1853],
        [0.4887, 0.2381, 0.7935],
        [0.2123, 0.7004, 0.5028],
        [0.9155, 0.8644, 0.4475],
        [0.8259, 0.9018, 0.2631]])

Tensor moved to GPU:
tensor([[0.7607, 0.6726, 0.1853],
        [0.4887, 0.2381, 0.7935],
        [0.2123, 0.7004, 0.5028],
        [0.9155, 0.8644, 0.4475],
        [0.8259, 0.9018, 0.2631]], device='cuda:0')

Result of operation on GPU (x * 2):
tensor([[1.5213, 1.3452, 0.3707],
        [0.9775, 0.4761, 1.5870],
        [0.4245, 1.4008, 1.0056],
        [1.8310, 1.7288, 0.8951],
        [1.6519, 1.8035, 0.5263]], device='cuda:0')

✅ Basic GPU tensor operations successful!


### 自动切换设备
在每一个 Notebook 开头，只需要复制并运行以下这段标准代码，就能自动适配当前环境（无论是本地 CPU、Colab GPU 还是服务器）：

```python
import torch

# 1. 自动检测并设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. 为 HF Pipeline 准备的设备 ID (GPU=0, CPU=-1)
pipeline_device = 0 if torch.cuda.is_available() else -1
```

### 之后如何使用？

1.  **对于 PyTorch 张量/模型**：
    使用 `.to(device)` 方法。
    ```python
    my_tensor = torch.randn(2, 2).to(device)
    my_model = MyModel().to(device)
    ```

2.  **对于 Hugging Face `pipeline`**：
    使用 `device` 参数（传入整数 ID）。
    ```python
    # 自动在 GPU 上运行（如果有的话）
    classifier = pipeline("sentiment-analysis", device=pipeline_device)
    ```

3.  **对于 `AutoModel` / `Trainer`**：
    *   `AutoModel` 加载后需手动 `.to(device)`。
    *   `Trainer` 通常会自动检测 GPU，无需手动干预。
